In [1]:
from common import *
from database import get_engine, run_query
from sqlalchemy import text
engine = get_engine()

In [2]:
sql_order_features = """
DROP TABLE IF EXISTS order_features;
CREATE TABLE order_features AS
WITH items_agg AS (
    SELECT order_id,
           SUM(price) AS items_price,
           SUM(freight_value) AS freight_total,
           SUM(price + freight_value) AS total_order_value,
           COUNT(*) AS item_count
    FROM order_items
    GROUP BY order_id
),
pay_agg AS (
    SELECT order_id, SUM(payment_value) AS payment_total
    FROM order_payments
    GROUP BY order_id
)
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    i.total_order_value,
    i.item_count,
    p.payment_total,
    DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) AS delivery_days,
    DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) AS delivery_delay_days,
    CASE
        WHEN o.order_delivered_customer_date IS NULL THEN NULL
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1
        ELSE 0
    END AS is_late
FROM orders o
LEFT JOIN items_agg i ON i.order_id = o.order_id
LEFT JOIN pay_agg p ON p.order_id = o.order_id;
"""
with engine.begin() as conn:
    for stmt in sql_order_features.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
print("order_features created")

order_features created


In [3]:
sql_customer_features = """
DROP TABLE IF EXISTS customer_features;
CREATE TABLE customer_features AS
WITH order_level AS (
    SELECT c.customer_unique_id,
           f.order_id,
           f.total_order_value,
           f.order_purchase_timestamp
    FROM order_features f
    JOIN customers c ON c.customer_id = f.customer_id
    WHERE f.order_status != 'canceled'
)
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS customer_order_count,
    SUM(total_order_value) AS customer_total_spending,
    AVG(total_order_value) AS customer_avg_order_value,
    MIN(order_purchase_timestamp) AS first_order_date,
    MAX(order_purchase_timestamp) AS last_order_date,
    CASE WHEN COUNT(DISTINCT order_id) >= 2 THEN 1 ELSE 0 END AS is_repeat_customer
FROM order_level
GROUP BY customer_unique_id;
"""
with engine.begin() as conn:
    for stmt in sql_customer_features.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
print("customer_features created")

customer_features created


In [4]:
sql_seller_features = """
DROP TABLE IF EXISTS seller_features;
CREATE TABLE seller_features AS
WITH seller_orders AS (
    SELECT oi.seller_id,
           oi.order_id,
           oi.price + oi.freight_value AS item_value
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status != 'canceled'
),
seller_reviews AS (
    SELECT so.seller_id, AVG(r.review_score) AS avg_review_score
    FROM seller_orders so
    JOIN order_reviews r ON r.order_id = so.order_id
    GROUP BY so.seller_id
)
SELECT
    s.seller_id,
    COUNT(DISTINCT so.order_id) AS seller_order_count,
    SUM(so.item_value) AS seller_revenue,
    sr.avg_review_score
FROM sellers s
LEFT JOIN seller_orders so ON so.seller_id = s.seller_id
LEFT JOIN seller_reviews sr ON sr.seller_id = s.seller_id
GROUP BY s.seller_id, sr.avg_review_score;
"""
with engine.begin() as conn:
    for stmt in sql_seller_features.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
print("seller_features created")

seller_features created
